In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from itertools import product
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

def calculate_revenue_import_correlation(db_info, start_date, end_date,
                                        min_periods=8, save_path=None):
    """
    미국 기업 매출과 HS 코드별 수입액의 상관계수 분석

    Parameters:
    -----------
    db_info : dict
        데이터베이스 연결 정보
    start_date : str
        분석 시작일 (형식: 'YYYY-MM-DD')
    end_date : str
        분석 종료일 (형식: 'YYYY-MM-DD')
    min_periods : int
        최소 데이터 포인트 수 (기본값: 8)
    save_path : str, optional
        CSV 저장 경로

    Returns:
    --------
    pd.DataFrame
        Long format의 상관계수 데이터프레임
        컬럼: ticker, hs_code, correlation, p_value, n_periods, start_date, end_date
    """

    # SQLAlchemy 엔진 생성
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info.get('port', 3306)}/{db_info['database']}"
    )

    print("=" * 80)
    print("미국 기업 매출 vs HS 코드 수입액 상관관계 분석")
    print("=" * 80)
    print(f"\n📅 분석 기간: {start_date} ~ {end_date}")
    print(f"📊 최소 데이터 포인트: {min_periods}개")

    # 1. 미국 기업 매출 데이터 로드
    print("\n" + "=" * 80)
    print("Step 1: 미국 기업 매출 데이터 로드")
    print("=" * 80)

    revenue_query = f"""
    SELECT
        ticker,
        date,
        value as revenue
    FROM investarus_sec_financial_data
    WHERE item_name = 'Revenues'
        AND date >= '{start_date}'
        AND date <= '{end_date}'
        AND value IS NOT NULL
        AND value > 0
    ORDER BY ticker, date
    """

    print("💾 매출 데이터 조회 중...")
    revenue_df = pd.read_sql(revenue_query, engine)
    revenue_df['date'] = pd.to_datetime(revenue_df['date'])

    print(f"✓ 총 레코드: {len(revenue_df):,}")
    print(f"✓ 기업 수: {revenue_df['ticker'].nunique():,}")
    print(f"✓ 날짜 범위: {revenue_df['date'].min()} ~ {revenue_df['date'].max()}")

    # 2. HS 코드 수입 데이터 로드
    print("\n" + "=" * 80)
    print("Step 2: HS 코드 수입 데이터 로드")
    print("=" * 80)

    import_query = f"""
    SELECT
        hs_code_6d as hs_code,
        date,
        impDlr as import_value
    FROM investarus_us_trade_import_quarter_with_forecast
    WHERE date >= '{start_date}'
        AND date <= '{end_date}'
        AND impDlr IS NOT NULL
        AND impDlr > 0
        AND forecast_flag = 0
    ORDER BY hs_code_6d, date
    """

    print("💾 수입 데이터 조회 중...")
    import_df = pd.read_sql(import_query, engine)
    import_df['date'] = pd.to_datetime(import_df['date'])

    print(f"✓ 총 레코드: {len(import_df):,}")
    print(f"✓ HS 코드 수: {import_df['hs_code'].nunique():,}")
    print(f"✓ 날짜 범위: {import_df['date'].min()} ~ {import_df['date'].max()}")

    engine.dispose()

    # 3. 날짜를 분기로 표준화
    print("\n" + "=" * 80)
    print("Step 3: 날짜 표준화 (분기 단위)")
    print("=" * 80)

    revenue_df['quarter'] = revenue_df['date'].dt.to_period('Q')
    import_df['quarter'] = import_df['date'].dt.to_period('Q')

    print("✓ 날짜를 분기로 변환 완료")

    # 4. 분기별 집계 (중복 데이터 처리)
    print("\n📊 분기별 데이터 집계 중...")

    # 매출 데이터: 분기별 평균 (또는 합계)
    revenue_pivot = revenue_df.groupby(['ticker', 'quarter'])['revenue'].mean().reset_index()
    revenue_pivot = revenue_pivot.pivot(index='quarter', columns='ticker', values='revenue')

    # 수입 데이터: 분기별 평균 (또는 합계)
    import_pivot = import_df.groupby(['hs_code', 'quarter'])['import_value'].mean().reset_index()
    import_pivot = import_pivot.pivot(index='quarter', columns='hs_code', values='import_value')

    print(f"✓ 매출 데이터 shape: {revenue_pivot.shape}")
    print(f"✓ 수입 데이터 shape: {import_pivot.shape}")

    # 5. 상관계수 계산
    print("\n" + "=" * 80)
    print("Step 4: 상관계수 계산")
    print("=" * 80)

    tickers = revenue_pivot.columns.tolist()
    hs_codes = import_pivot.columns.tolist()

    total_combinations = len(tickers) * len(hs_codes)
    print(f"📊 총 조합 수: {len(tickers):,} tickers × {len(hs_codes):,} HS codes = {total_combinations:,}")

    results = []
    processed = 0
    valid_correlations = 0

    print("\n💫 상관계수 계산 중...")

    for ticker in tickers:
        for hs_code in hs_codes:
            processed += 1

            # 진행상황 표시
            if processed % 10000 == 0:
                print(f"  진행률: {processed:,}/{total_combinations:,} ({processed/total_combinations*100:.1f}%)")

            # 두 시계열 데이터 추출
            revenue_series = revenue_pivot[ticker]
            import_series = import_pivot[hs_code]

            # 공통 분기 찾기
            common_quarters = revenue_series.index.intersection(import_series.index)

            if len(common_quarters) < min_periods:
                continue

            # 공통 분기 데이터만 선택
            rev_values = revenue_series.loc[common_quarters].values
            imp_values = import_series.loc[common_quarters].values

            # 결측치 제거
            valid_mask = ~(np.isnan(rev_values) | np.isnan(imp_values))
            rev_values = rev_values[valid_mask]
            imp_values = imp_values[valid_mask]

            if len(rev_values) < min_periods:
                continue

            # 상관계수 계산
            try:
                correlation, p_value = pearsonr(rev_values, imp_values)

                # 유효한 상관계수인지 확인
                if not np.isnan(correlation):
                    results.append({
                        'ticker': ticker,
                        'hs_code': hs_code,
                        'correlation': correlation,
                        'p_value': p_value,
                        'n_periods': len(rev_values),
                        'start_quarter': str(common_quarters.min()),
                        'end_quarter': str(common_quarters.max())
                    })
                    valid_correlations += 1

            except Exception as e:
                continue

    print(f"\n✓ 계산 완료: {processed:,}개 조합 처리")
    print(f"✓ 유효한 상관계수: {valid_correlations:,}개")

    # 6. 결과 데이터프레임 생성
    print("\n" + "=" * 80)
    print("Step 5: 결과 정리")
    print("=" * 80)

    if len(results) == 0:
        print("⚠️ 유효한 상관계수가 없습니다.")
        return pd.DataFrame()

    result_df = pd.DataFrame(results)

    # 날짜 정보 추가
    result_df.insert(0, 'analysis_start_date', start_date)
    result_df.insert(1, 'analysis_end_date', end_date)

    # 상관계수 절댓값 기준 내림차순 정렬
    result_df['abs_correlation'] = result_df['correlation'].abs()
    result_df = result_df.sort_values('abs_correlation', ascending=False)
    result_df = result_df.drop('abs_correlation', axis=1)
    result_df = result_df.reset_index(drop=True)

    # 통계 정보
    print(f"\n📊 상관계수 통계:")
    print(f"  평균: {result_df['correlation'].mean():.4f}")
    print(f"  중앙값: {result_df['correlation'].median():.4f}")
    print(f"  표준편차: {result_df['correlation'].std():.4f}")
    print(f"  최대값: {result_df['correlation'].max():.4f}")
    print(f"  최소값: {result_df['correlation'].min():.4f}")

    # 상관계수 구간별 분포
    strong_positive = (result_df['correlation'] >= 0.7).sum()
    moderate_positive = ((result_df['correlation'] >= 0.4) & (result_df['correlation'] < 0.7)).sum()
    weak_positive = ((result_df['correlation'] >= 0.1) & (result_df['correlation'] < 0.4)).sum()
    weak_negative = ((result_df['correlation'] > -0.4) & (result_df['correlation'] < -0.1)).sum()
    moderate_negative = ((result_df['correlation'] > -0.7) & (result_df['correlation'] <= -0.4)).sum()
    strong_negative = (result_df['correlation'] <= -0.7).sum()

    print(f"\n📈 상관계수 분포:")
    print(f"  강한 양의 상관 (≥0.7): {strong_positive:,}개 ({strong_positive/len(result_df)*100:.1f}%)")
    print(f"  중간 양의 상관 (0.4~0.7): {moderate_positive:,}개 ({moderate_positive/len(result_df)*100:.1f}%)")
    print(f"  약한 양의 상관 (0.1~0.4): {weak_positive:,}개 ({weak_positive/len(result_df)*100:.1f}%)")
    print(f"  약한 음의 상관 (-0.4~-0.1): {weak_negative:,}개 ({weak_negative/len(result_df)*100:.1f}%)")
    print(f"  중간 음의 상관 (-0.7~-0.4): {moderate_negative:,}개 ({moderate_negative/len(result_df)*100:.1f}%)")
    print(f"  강한 음의 상관 (≤-0.7): {strong_negative:,}개 ({strong_negative/len(result_df)*100:.1f}%)")

    # 유의수준별 분포
    sig_001 = (result_df['p_value'] < 0.01).sum()
    sig_005 = ((result_df['p_value'] >= 0.01) & (result_df['p_value'] < 0.05)).sum()
    sig_010 = ((result_df['p_value'] >= 0.05) & (result_df['p_value'] < 0.10)).sum()

    print(f"\n📉 통계적 유의성:")
    print(f"  p < 0.01: {sig_001:,}개 ({sig_001/len(result_df)*100:.1f}%)")
    print(f"  0.01 ≤ p < 0.05: {sig_005:,}개 ({sig_005/len(result_df)*100:.1f}%)")
    print(f"  0.05 ≤ p < 0.10: {sig_010:,}개 ({sig_010/len(result_df)*100:.1f}%)")

    # 상위 결과 출력
    print(f"\n🔝 상위 10개 상관관계 (절댓값 기준):")
    display_df = result_df.head(10).copy()
    display_df['correlation'] = display_df['correlation'].apply(lambda x: f"{x:.4f}")
    display_df['p_value'] = display_df['p_value'].apply(lambda x: f"{x:.4e}")
    print(display_df[['ticker', 'hs_code', 'correlation', 'p_value', 'n_periods']].to_string(index=False))

    # 7. CSV 파일 저장
    if save_path:
        import os
        os.makedirs(save_path, exist_ok=True)

        start_str = start_date.replace('-', '')
        end_str = end_date.replace('-', '')
        filename = f"revenue_import_correlation_{start_str}_to_{end_str}.csv"
        filepath = os.path.join(save_path, filename)

        result_df.to_csv(filepath, index=False, encoding='utf-8-sig')
        print(f"\n💾 결과 저장 완료:")
        print(f"   {filepath}")

    print("\n" + "=" * 80)
    print("분석 완료")
    print("=" * 80)

    return result_df


def get_top_correlations(correlation_df, ticker=None, hs_code=None,
                        top_n=10, min_correlation=None):
    """
    특정 조건의 상위 상관관계 추출

    Parameters:
    -----------
    correlation_df : pd.DataFrame
        상관계수 데이터프레임
    ticker : str, optional
        특정 기업 티커
    hs_code : str, optional
        특정 HS 코드
    top_n : int
        상위 N개 추출
    min_correlation : float, optional
        최소 상관계수 절댓값

    Returns:
    --------
    pd.DataFrame
        필터링된 상관계수 데이터프레임
    """

    df = correlation_df.copy()

    # 필터링
    if ticker:
        df = df[df['ticker'] == ticker]

    if hs_code:
        df = df[df['hs_code'] == hs_code]

    if min_correlation:
        df = df[df['correlation'].abs() >= min_correlation]

    # 상관계수 절댓값 기준 정렬
    df['abs_correlation'] = df['correlation'].abs()
    df = df.sort_values('abs_correlation', ascending=False)
    df = df.drop('abs_correlation', axis=1)

    return df.head(top_n).reset_index(drop=True)


# 사용 예시
# if __name__ == "__main__":
#     # 데이터베이스 연결 정보
#     db_info = {
#         'host': 'localhost',
#         'user': 'your_username',
#         'password': 'your_password',
#         'database': 'your_database',
#         'port': 3306
#     }
#
#     # 저장 경로
#     save_path = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\correlation"
#
#     # 상관계수 분석 실행
#     correlation_df = calculate_revenue_import_correlation(
#         db_info=db_info,
#         start_date='2013-01-01',
#         end_date='2023-12-31',
#         min_periods=8,  # 최소 8개 분기 데이터 필요
#         save_path=save_path
#     )
#
#     # 특정 기업의 상위 상관관계 조회
#     if not correlation_df.empty:
#         print("\n" + "=" * 80)
#         print("AAPL의 상위 10개 상관관계")
#         print("=" * 80)
#         aapl_corr = get_top_correlations(
#             correlation_df,
#             ticker='AAPL',
#             top_n=10
#         )
#         print(aapl_corr)
#
#         # 특정 HS 코드의 상위 상관관계 조회
#         print("\n" + "=" * 80)
#         print("HS Code 100630의 상위 10개 상관관계")
#         print("=" * 80)
#         hs_corr = get_top_correlations(
#             correlation_df,
#             hs_code='100630',
#             top_n=10
#         )
#         print(hs_corr)
#
#         # 강한 상관관계만 필터링 (|r| >= 0.7)
#         print("\n" + "=" * 80)
#         print("강한 상관관계 (|r| ≥ 0.7)")
#         print("=" * 80)
#         strong_corr = get_top_correlations(
#             correlation_df,
#             min_correlation=0.7,
#             top_n=20
#         )
#         print(strong_corr)
# ```
#
# ## 주요 기능
#
# ### 1. 데이터 처리
# - 매출 데이터: `item_name = 'Revenues'`인 데이터만 추출
# - 수입 데이터: `forecast_flag = 0`인 실적 데이터만 사용
# - 분기 단위로 날짜 표준화
#
# ### 2. 상관계수 계산
# - Pearson 상관계수 및 p-value 계산
# - 최소 데이터 포인트 수 설정 가능
# - 공통 분기만 사용하여 계산
#
# ### 3. Long Format 출력
# ```
# analysis_start_date | analysis_end_date | ticker | hs_code | correlation | p_value | n_periods | start_quarter | end_quarter

In [ ]:
from DATA.stock_invest_function import get_db_host

db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}

save_path = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\US_trade_revenue_corr"

# 전체 상관계수 분석
correlation_df = calculate_revenue_import_correlation(
    db_info=db_info,
    start_date='2013-01-01',
    end_date='2023-12-31',
    min_periods=8,
    save_path=save_path
)

# # 특정 기업의 상관관계
# aapl_corr = get_top_correlations(correlation_df, ticker='AAPL', top_n=10)
#
# # 특정 HS 코드의 상관관계
# hs_corr = get_top_correlations(correlation_df, hs_code='100630', top_n=10)
#
# # 강한 상관관계만 필터링
# strong_corr = get_top_correlations(correlation_df, min_correlation=0.7, top_n=20)